In [ ]:
from pyspark.sql import functions as F

import uuid
import time
import json
from faker import Faker

fake = Faker()

In [ ]:
CATALOG = "lakehouse_dev"
SCHEMA = "cloudwatch_metrics_pull"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/prod"
USER_LANDING = f"{BASE_PATH}/users_json"
SPEND_LANDING = f"{BASE_PATH}/spend_csv"
csv_schema = "id,age,annual_income,spending_core"

In [ ]:
def generate_coordinated_events(batch_size=5):
    user_events = []
    spend_events = []
    shared_ids = [fake.random_int(min=1, max=1000) for _ in range(batch_size)]
    spend_events.append(csv_schema)

    for user_id in shared_ids:
        user_events.append({
            "id": user_id,
            "email": fake.email(),
            "creation_date": fake.date_time_this_year().strftime("%m-%d-%Y %H:%M:%S"),
            "last_activity_date": fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"),
            "firstname": fake.first_name(),
            "lastname": fake.last_name(),
            "address": fake.address().replace("\n", ", "),
            "city": fake.city(),
            "last_ip": fake.ipv4(),
            "postcode": fake.postcode()
        })
        spend_events.append(
            f"{user_id},{fake.random_int(18, 90)},{round(fake.pyfloat(left_digits=3, right_digits=1, positive=True), 1)},{round(fake.pyfloat(left_digits=2, right_digits=1, positive=True), 1)}"
        )
    return user_events, spend_events


def run_simulation(batch_size=10):
    user_data, spend_data = generate_coordinated_events(batch_size)
    unique_id = uuid.uuid4().hex[:8]
    timestamp = int(time.time())

    # Write User JSON (Newline Delimited)
    user_json_str = "\n".join([json.dumps(e) for e in user_data])
    user_file = f"{USER_LANDING}/sim_users_{timestamp}_{unique_id}.json"
    dbutils.fs.put(user_file, user_json_str, overwrite=True)

    # Write Spend CSV (No header, matching the raw ingestion pattern)
    spend_csv_str = "\n".join(spend_data)
    spend_file = f"{SPEND_LANDING}/sim_spend_{timestamp}_{unique_id}.csv"
    dbutils.fs.put(spend_file, spend_csv_str, overwrite=True)

    print(f"Simulation Aligned with Test Data:")
    print(f" - Produced {batch_size} coordinated records.")
    print(f" - Users path: {user_file}")
    print(f" - Spend path: {spend_file}")

# List of directories to clean
directories = [
    f"{BASE_PATH}/users_json",
    f"{BASE_PATH}/spend_csv"
]

def cleanup_landing_zone():
    for directory in directories:
        try:
            # recurse=True ensures all files and subdirectories are removed
            dbutils.fs.rm(directory, recurse=True)
            # Recreate the directory so the simulation script doesn't fail on missing paths
            dbutils.fs.mkdirs(directory)
            print(f"✅ Cleaned and recreated: {directory}")
        except Exception as e:
            print(f"⚠️ Error cleaning {directory}: {e}")




## ⚠️ Cleanup PROD LANDING ZONE ⚠️

In [ ]:
cleanup_landing_zone()

## ⚠️ Clean up all tables ⚠️

In [ ]:
def drop_medallion_tables():
    tables = [
        "user_bronze_sdp",
        "raw_user_data",
        "raw_spend_data",
        "user_silver_sdp",
        "spend_silver_sdp",
        "user_gold_sdp"
    ]

    print(f"Starting cleanup for schema: {CATALOG}.{SCHEMA}")

    for table in tables:
        table_path = f"{CATALOG}.{SCHEMA}.{table}"
        try:
            # Use IF EXISTS to prevent errors if a table hasn't been created yet
            spark.sql(f"DROP TABLE IF EXISTS {table_path}")
            print(f"✅ Successfully dropped table: {table_path}")
        except Exception as e:
            print(f"⚠️ Failed to drop {table_path}: {str(e)}")


drop_medallion_tables()

## Produce new data

In [ ]:
run_simulation(5)

## Check Landing Zone for new files

In [ ]:
print("Checking Landing Zone Volumes...")
user_files = dbutils.fs.ls(f"{USER_LANDING}")
spend_files = dbutils.fs.ls(f"{SPEND_LANDING}")

print(f"Found {len(user_files)} user files and {len(spend_files)} spend files.")

# 2. Check Processing Status via Table Metadata
# This shows you the 'last_modified' time and row counts
tables = ["user_bronze_sdp", "spend_silver_sdp", "user_gold_sdp"]

for table in tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        df = spark.table(full_name)
        count = df.count()
        # Get the latest record based on your simulated timestamps
        latest = df.select(F.max("_metadata.file_modification_time")).collect()[0][0] if "_metadata" in df.columns else "N/A"
        print(f"Table: {table} | Row Count: {count} | Latest File Processed: {latest}")
    except Exception as e:
        print(f"Could not read {full_name}: {e}")

# 3. Validation: Match User IDs between Bronze and Gold
# This confirms the join in 03-gold.py worked for your Faker data
bronze_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_bronze_sdp").select("id").distinct()
gold_ids = spark.table(f"{CATALOG}.{SCHEMA}.user_gold_sdp").select("id").distinct()

match_count = gold_ids.intersect(bronze_ids).count()
print(f"Successful Joins in Gold Layer: {match_count}")

## ALL

In [ ]:
%sql
SELECT
  timestamp,
  -- Use dot notation to reach into the struct and array
  error.exceptions[0].message AS error_detail,

  -- Extract the specific rule name from the message string
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,

  -- Extract the input data snippet
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,

  origin.pipeline_id
FROM event_log("ab370c25-0a6c-4350-8db5-3a5f1d8eac26")
WHERE level = 'ERROR'
  -- Use the size function to ensure an exception actually exists before checking the message
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message IS NOT NULL
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

## ALL WITHIN 20 MIN WINDOW

In [ ]:
%sql
SELECT
  timestamp,
  -- Extracting the specific rule name
  regexp_extract(error.exceptions[0].message, "Violated expectations: '([^']+)'", 1) AS violated_rule,
  -- Extracting the record snippet for the CloudWatch log
  regexp_extract(error.exceptions[0].message, "Input data: '(.+)'", 1) AS offending_record,
  origin.pipeline_id,
  origin.update_id
FROM event_log("ab370c25-0a6c-4350-8db5-3a5f1d8eac26")
WHERE level = 'ERROR'
  AND timestamp >= (current_timestamp() - INTERVAL 20 MINUTES)
  AND size(error.exceptions) > 0
  AND error.exceptions[0].message LIKE '%failed to meet the expectation%'
ORDER BY timestamp DESC

### Adding the new rule to your expectations metadata table

In [ ]:
%sql
INSERT INTO lakehouse_dev.cloudwatch_metrics_pull.expectations (tag, name, constraint)
VALUES (
  'user_silver_sdp',
  'valid_activity_sequence',
  'last_activity_date >= creation_date'
);